# GroundX × NVIDIA — 5-Minute Quickstart

Ask questions of complex enterprise documents and get **page-cited answers** — powered by GroundX document intelligence, queryable by any agent stack (shown here raw, via the Python SDK, and via the NeMo Agent Toolkit config in this repo).

**Data flow disclosure:** this notebook talks to the GroundX hosted service (`api.groundx.ai`). Documents you upload in Act 2 go to your GroundX account and can be deleted at any time (cell at the bottom). Nothing here sends your documents to NVIDIA — an agent's LLM only ever sees retrieved text passages. For fully self-hosted operation (air-gapped capable), see `docs/it-reviewer-fact-sheet.md`.

**Keys:** copy `.env.example` to `.env` and add `GROUNDX_API_KEY` (free at dashboard.eyelevel.ai) and, for the agent demo, `NVIDIA_API_KEY` (free at build.nvidia.com).

In [ ]:
# Setup — takes seconds
%pip -q install groundx python-dotenv
from dotenv import load_dotenv; load_dotenv()
import os
from groundx import GroundX
gx = GroundX(api_key=os.environ["GROUNDX_API_KEY"])
print("connected")

## Act 1 — the wow moment (pre-ingested corpus)

The demo bucket already contains the IRS Form 1040 instructions — 100+ pages of dense tables. Ask a hard question and note the answer comes back with the **exact page and bounding box** of the source.

In [ ]:
BUCKET_NAME = os.environ.get("COLLECTION_NAME", "nvidia-quickstart-demo")
bucket = next(b for b in gx.buckets.list().buckets if b.name == BUCKET_NAME)

r = gx.search.content(id=bucket.bucket_id, query="What is the standard deduction for married filing jointly?", n=3)
top = r.search.results[0]
print("file: ", top.file_name)
print("page: ", top.bounding_boxes[0].page_number if top.bounding_boxes else "?")
print("box:  ", top.bounding_boxes[0] if top.bounding_boxes else "?")
print()
print((top.suggested_text or top.text or "")[:600])

Every result is traceable to a rectangle on a page — that's what makes answers *auditable*, which is the difference between a demo and something a regulated business can deploy.

## Act 2 — bring your own document

Point the next cell at any PDF URL (or adapt for a local file with `gx.ingest`). Full agentic processing of a short document takes a few minutes.

In [ ]:
import time
DOC_URL = "https://www.irs.gov/pub/irs-pdf/fw9.pdf"   # <-- replace with your document URL

ing = gx.ingest(documents=[{
    "bucket_id": bucket.bucket_id, "file_name": "my-document.pdf",
    "file_type": "pdf", "source_url": DOC_URL, "process_level": "full"}])
pid = ing.ingest.process_id
while True:
    st = gx.documents.get_processing_status_by_id(process_id=pid).ingest.status
    print("status:", st)
    if st in ("complete", "error", "cancelled"): break
    time.sleep(20)

In [ ]:
r = gx.search.content(id=bucket.bucket_id, query="When is a payee exempt from backup withholding?", n=3)
for res in r.search.results[:2]:
    page = res.bounding_boxes[0].page_number if res.bounding_boxes else "?"
    print(f"[{res.file_name} p.{page}]", (res.suggested_text or res.text or "")[:200], "\n")

## Act 3 — raw API (for the API-first reviewer)

No SDK required — one POST with the key in a transport header.

In [ ]:
import requests
resp = requests.post(
    f"https://api.groundx.ai/api/v1/search/{bucket.bucket_id}",
    headers={"X-API-Key": os.environ["GROUNDX_API_KEY"]},
    json={"query": "standard deduction amounts by filing status", "n": 2}, timeout=60)
print(resp.status_code, "| first result page:",
      resp.json()["search"]["results"][0]["boundingBoxes"][0]["pageNumber"])

## Act 4 — the agent (NeMo Agent Toolkit + Nemotron)

The same corpus, driven by NVIDIA's agent stack — config in [`configs/groundx_agent.yml`](../configs/groundx_agent.yml), zero glue code:

```bash
pip install "nvidia-nat[langchain,mcp]"
nat run --config_file configs/groundx_agent.yml \
  --input "Search the bucket named nvidia-quickstart-demo and answer: what is the standard deduction for married filing jointly? Cite the page."
```

Verified output: *"The standard deduction for married filing jointly in 2025 is $31,500 … (page 35)."*

## Cleanup

Delete anything you uploaded:

In [ ]:
# for d in gx.documents.lookup(id=bucket.bucket_id).documents:
#     if d.file_name == "my-document.pdf":
#         gx.documents.delete(document_ids=[d.document_id])
print("uncomment to delete your uploads")